# State Electricity Generation Analysis, 1990-2024

How has each U.S. state changed their green energy generation over the last three decades?

This notebook takes net generation data from the EIA, categorizes each fuel source as green or non-green, 
and shows how each U.S. state has shifted toward green electricity from 1990 to 2024. 

The data output is fed to a Power BI dashboard to visualize the information.

**Green energy** (or renewable energy) is considered: Nuclear, Hydroelectric, Wind, Solar, Geothermal,
Wood, and Other Biomass. 

**Non-green energy** is considered: Coal, Natural Gas, Petroleum, and Other Gases. (Note: biomass is counted as green by renewability, but it does emit CO2)

Data source: EIA, Annual Net Generation by State
(https://www.eia.gov/electricity/data/state/)

## 1. Load the data

The EIA provided data has a junk banner on the first row.

In [ ]:
import pandas as pd

raw = pd.read_excel("annual_generation_state.xls", header=None, nrows=10)

for i, row in raw.iterrows():
    print(i, list(row.values))

The real header starts at row 1.

In [ ]:
gen = pd.read_excel("annual_generation_state.xls", skiprows=1)

gen.info()
gen.head()

## 2. Explore the categories

Before filtering, verify the column data.

In [ ]:
print(gen["YEAR"].min(), "to", gen["YEAR"].max())
print()
print(gen["TYPE OF PRODUCER"].unique())
print()
print(gen["ENERGY SOURCE"].unique())

## 3. Filter to one energy producer type

The EIA provided data breaks generation down by producer type (utilities, independent producers, etc).
Keep only the "Total Electric Power Industry" to avoid double counting.

In [ ]:
gen = gen[gen["TYPE OF PRODUCER"] == "Total Electric Power Industry"]

print(len(gen), "rows remaining")

This is a state analysis, so drop any non-state or blank state codes.

In [ ]:
gen = gen[~gen["STATE"].isin(["US-TOTAL", "US-Total", "  ", " "])]

print(sorted(gen["STATE"].unique()))
print(len(gen["STATE"].unique()), "states (50 + DC)")

## 4. Separate the per-fuel rows

Keep only the "per-fuel" observations to categorize them later. Exclude Pumped Storage and Other as they are not relevant to the analysis.

In [ ]:
totals = gen[gen["ENERGY SOURCE"] == "Total"]
fuels = gen[gen["ENERGY SOURCE"] != "Total"]

print(len(fuels), "fuel rows")
print(sorted(fuels["ENERGY SOURCE"].unique()))

fuels = fuels[~fuels["ENERGY SOURCE"].isin(["Pumped Storage", "Other"])]

print(sorted(fuels["ENERGY SOURCE"].unique()))

## 5. Categorize green vs non-green

Categorization of the energy generation sources. Everything in `green_sources` is green; everything else is non-green. This can be adjusted to your liking.

In [ ]:
green_sources = [
    "Nuclear",
    "Hydroelectric Conventional",
    "Wind",
    "Solar Thermal and Photovoltaic",
    "Geothermal",
    "Wood and Wood Derived Fuels",
    "Other Biomass",
]

fuels["CATEGORY"] = fuels["ENERGY SOURCE"].apply(
    lambda x: "Green" if x in green_sources else "Non-Green"
)

# Verify the categorization
print(fuels.groupby("CATEGORY")["ENERGY SOURCE"].unique())

## 6. Build the state-year table

Group by year, state, and category to sum generation, then pivot so each state-year is one
row with Green and Non-Green columns. Add TOTAL and GREEN_SHARE.

In [ ]:
grouped = (
    fuels.groupby(["YEAR", "STATE", "CATEGORY"])["GENERATION (Megawatthours)"]
    .sum()
    .reset_index()
)

pivot = grouped.pivot_table(
    index=["YEAR", "STATE"],
    columns="CATEGORY",
    values="GENERATION (Megawatthours)",
)

pivot.head()

There are a few instances of NaN datapoints where no green energy was produced for a given year. Convert NaN to 0.

In [ ]:
pivot["Green"] = pivot["Green"].fillna(0)
pivot["Non-Green"] = pivot["Non-Green"].fillna(0)

pivot["TOTAL"] = pivot["Green"] + pivot["Non-Green"]
pivot["GREEN_SHARE"] = pivot["Green"] / pivot["TOTAL"]

pivot = pivot.reset_index()
pivot.columns.name = None

pivot.head()

## 7. Analysis #1 - Largest Changes in Green Energy Share

Which states increased the green *share* of their grid the most?

FYI, green energy share = percentage of green energy generation relative to total energy generation.

In [ ]:
start = pivot[pivot["YEAR"] == 1990][["STATE", "GREEN_SHARE"]]
end = pivot[pivot["YEAR"] == 2024][["STATE", "GREEN_SHARE"]]

change = start.merge(end, on="STATE", suffixes=("_1990", "_2024"))
change["CHANGE"] = change["GREEN_SHARE_2024"] - change["GREEN_SHARE_1990"]
change = change.sort_values("CHANGE", ascending=False)

change.head(15).style.format({
    "GREEN_SHARE_1990": "{:.1%}",
    "GREEN_SHARE_2024": "{:.1%}",
    "CHANGE": "{:.1%}",
})

Iowa leads, driven by its Wind investments. However, D.C is also in the top, rising from 0% to
~47% over the course of 3 deacdes. D.C is relatively tiny to the other states, so a small amount of solar swings its entire percentage. This is the
limitation of a share-based metric: it over-rewards small states. Hence, a second analysis is needed.

In [ ]:
import matplotlib.pyplot as plt

top15 = change.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top15["STATE"], top15["CHANGE"] * 100, color="#2ca02c")
plt.xlabel("Change in green share (percentage points), 1990-2024")
plt.ylabel("State")
plt.title("States that greened their electricity most (by share)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Analysis #2 - Absolute Green Energy (MWh) added

A more honest "impact" metric is how much green energy generation in MWh a state added over the past 3 decades. This can be calculated by taking the green energy generated in 2024 minus the green energy generated in 1990.

In [ ]:
green_start = pivot[pivot["YEAR"] == 1990][["STATE", "Green"]]
green_end = pivot[pivot["YEAR"] == 2024][["STATE", "Green"]]

green_change = green_start.merge(green_end, on="STATE", suffixes=("_1990", "_2024"))
green_change["GREEN_ADDED"] = green_change["Green_2024"] - green_change["Green_1990"]
green_change = green_change.sort_values("GREEN_ADDED", ascending=False)

green_change.head(15)

From the results, we see that Texas dominates, adding ~186 million MWh of green energy generation, mostly wind. The picture
is now completely different: big states win on absolute impact, small
states win on proportional change.

Let's take a look at the bottom 6 states:

In [ ]:
green_change.sort_values("GREEN_ADDED").head(6)

Washington, Oregon, and Maine among others all reduced their green energy generation. These are hydro and nuclear
heavy states - their green energy output was already high in 1990 and has been flat or declining,
while their total generation grew. So their green energy *share* decreased even though they were never heavily invested in non-green energy generation sources.

In [ ]:
top15 = green_change.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top15["STATE"], top15["GREEN_ADDED"] / 1_000_000, color="#2ca02c")
plt.xlabel("Green generation added (million MWh), 1990-2024")
plt.ylabel("State")
plt.title("States that added the most clean electricity (absolute)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Analysis #3 - Green Energy Share over Time

The two analyses are snapshots. Let's show the yearly trajectory for a few standout states:
Texas (wind boom), Iowa (proportional leader), California (steady growth), West Virginia
(coal country, stays low), and Connecticut (nuclear-heavy, declines).

In [ ]:
states_to_plot = ["TX", "IA", "CT", "WV", "CA"]

plt.figure(figsize=(11, 6))

for st in states_to_plot:
    data = pivot[pivot["STATE"] == st].sort_values("YEAR")
    plt.plot(data["YEAR"], data["GREEN_SHARE"] * 100, label=st, linewidth=2)

plt.xlabel("Year")
plt.ylabel("Green share of generation (%)")
plt.title("Green electricity share over time")
plt.legend(title="State")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Export the cleaned data

Save the full state-year table for use in the Power BI dashboard (map + time series visuals).

In [ ]:
pivot.to_csv("clean_state_green.csv", index=False)
print("Saved clean_state_green.csv:", pivot.shape)

## 11. Build the per-state summary table

The dashboard's ranking charts need one row per state. Build a summary
with the two analyses plus a third, size-adjusted metric.

In [ ]:
# Index by state so the columns align on subtraction
y1990 = pivot[pivot["YEAR"] == 1990].set_index("STATE")
y2024 = pivot[pivot["YEAR"] == 2024].set_index("STATE")

summary = pd.DataFrame({
    "PP_CHANGE": y2024["GREEN_SHARE"] - y1990["GREEN_SHARE"],
    "MWH_ADDED": y2024["Green"] - y1990["Green"],
    "SHARE_1990": y1990["GREEN_SHARE"],
    "SHARE_2024": y2024["GREEN_SHARE"],
})

#  Green energy added relative to total energy generation
summary["MWH_ADDED_PER_TOTAL"] = summary["MWH_ADDED"] / y2024["TOTAL"]

summary.sort_values("MWH_ADDED", ascending=False).head()

It's worth keeping in mind the metric (`MWH_ADDED_PER_TOTAL`) deflates the big-state effect, but
it introduces its own issue. States whose total generation *decreased* (like Vermont after its
nuclear plant closed in 2014) received an inflated ratio as a result, because the denominator became smaller. So a
single straight-forward metric doesn't encompass the whole story. Each analysis reveals a different conclusion, and together these findings paint the bigger picture
of how each U.S. state is adopting green energy generation sources.

In [ ]:
summary.to_csv("state_summary.csv", index=True)
print("Saved state_summary.csv:", summary.shape)

## Summary

Three analyses for three different conclusions:

- **By Green Energy Share Change:** Iowa, New Mexico, Kansas: proportional transformation, favors small grids
- **By Absolute MWh Added:** Texas, Illinois, California: real-world impact, favors large grids
- **By Size-Adjusted Added:** different again, but distorted by states whose total energy generation decreased (e.g., closing of nuclear plants)

The main conclusion is that *the metric you pick can influence who "wins"*.

Next steps: join EIA CO2 emissions data to test whether greener grids actually have lower emissions, and if not, why?